In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Postgres Star Schema ETL") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.4") \
    .getOrCreate()

pg_url = "jdbc:postgresql://postgres:5432/petshop_db"
pg_props = {"user": "admin", "password": "4321", "driver": "org.postgresql.Driver"}

print("Spark успешно запущен. Подключение к хосту 'postgres' настроено.")

Spark успешно запущен. Подключение к хосту 'postgres' настроено.


In [3]:
print("Читаем сырые данные из PostgreSQL...")
raw_df = spark.read.jdbc(url=pg_url, table="staging_raw_data", properties=pg_props)
raw_df.createOrReplaceTempView("raw_data")

print(f"Загружено строк: {raw_df.count()}")
spark.sql("SELECT c_email, p_name, order_total_price FROM raw_data LIMIT 5").show()

Читаем сырые данные из PostgreSQL...
Загружено строк: 10000
+--------------------+---------+--------------------+
|             c_email|   p_name|   order_total_price|
+--------------------+---------+--------------------+
|bmassingham0@army...| Dog Food|487.7000000000000...|
|  cscudder1@time.com|  Cat Toy|484.6100000000000...|
|  vhuxter2@fotki.com|Bird Cage|144.2400000000000...|
|bbullier3@bravesi...| Dog Food|441.0900000000000...|
| sedgar4@smugmug.com|Bird Cage|72.27000000000000...|
+--------------------+---------+--------------------+



In [4]:
print("Создаем и записываем Измерение: Клиенты...")
dim_customers = spark.sql("""
    SELECT ROW_NUMBER() OVER(ORDER BY c_email) AS customer_id, 
           MAX(c_first_name) as c_first_name, MAX(c_last_name) as c_last_name, MAX(c_age) as c_age, 
           c_email, 
           MAX(c_country) as c_country, MAX(c_zip) as c_zip, MAX(c_pet_type) as c_pet_type, 
           MAX(c_pet_name) as c_pet_name, MAX(c_pet_breed) as c_pet_breed
    FROM raw_data WHERE c_email IS NOT NULL
    GROUP BY c_email
""")
dim_customers.createOrReplaceTempView("dim_customers")
dim_customers.write.jdbc(url=pg_url, table="star_dim_customers", mode="overwrite", properties=pg_props)
print("Таблица star_dim_customers сохранена.")

Создаем и записываем Измерение: Клиенты...
Таблица star_dim_customers сохранена.


In [5]:
print("Создаем и записываем Измерение: Продукты...")
dim_products = spark.sql("""
    SELECT ROW_NUMBER() OVER(ORDER BY p_name, p_brand) AS product_id, 
           p_name, MAX(p_category) as p_category, MAX(animal_category) as animal_category, 
           MAX(p_price) as p_price, MAX(p_weight) as p_weight, MAX(p_color) as p_color, 
           MAX(p_size) as p_size, p_brand, MAX(p_material) as p_material, 
           MAX(p_rating) as p_rating, MAX(p_reviews_cnt) as p_reviews_cnt
    FROM raw_data WHERE p_name IS NOT NULL
    GROUP BY p_name, p_brand
""")
dim_products.createOrReplaceTempView("dim_products")
dim_products.write.jdbc(url=pg_url, table="star_dim_products", mode="overwrite", properties=pg_props)
print("Таблица star_dim_products сохранена.")

Создаем и записываем Измерение: Продукты...
Таблица star_dim_products сохранена.


In [6]:
print("Создаем и записываем Измерение: Магазины...")
dim_stores = spark.sql("""
    SELECT ROW_NUMBER() OVER(ORDER BY shop_name) AS store_id, 
           shop_name, MAX(shop_location) as shop_location, MAX(shop_city) as shop_city, 
           MAX(shop_state) as shop_state, MAX(shop_country) as shop_country, 
           MAX(shop_phone) as shop_phone, MAX(shop_email) as shop_email
    FROM raw_data WHERE shop_name IS NOT NULL
    GROUP BY shop_name
""")
dim_stores.createOrReplaceTempView("dim_stores")
dim_stores.write.jdbc(url=pg_url, table="star_dim_stores", mode="overwrite", properties=pg_props)
print("Таблица star_dim_stores сохранена.")

Создаем и записываем Измерение: Магазины...
Таблица star_dim_stores сохранена.


In [7]:
print("Создаем и записываем Измерение: Поставщики...")
dim_suppliers = spark.sql("""
    SELECT ROW_NUMBER() OVER(ORDER BY vendor_name) AS supplier_id, 
           vendor_name, MAX(vendor_contact) as vendor_contact, MAX(vendor_email) as vendor_email, 
           MAX(vendor_phone) as vendor_phone, MAX(vendor_address) as vendor_address, 
           MAX(vendor_city) as vendor_city, MAX(vendor_country) as vendor_country
    FROM raw_data WHERE vendor_name IS NOT NULL
    GROUP BY vendor_name
""")
dim_suppliers.createOrReplaceTempView("dim_suppliers")
dim_suppliers.write.jdbc(url=pg_url, table="star_dim_suppliers", mode="overwrite", properties=pg_props)
print("Таблица star_dim_suppliers сохранена.")

Создаем и записываем Измерение: Поставщики...
Таблица star_dim_suppliers сохранена.


In [8]:
print("Собираем Таблицу Фактов (star_fact_sales)...")
fact_sales = spark.sql("""
    SELECT 
        r.id AS sale_id, r.order_date AS sale_date, 
        c.customer_id, p.product_id, s.store_id, sup.supplier_id, 
        r.order_qty AS quantity, r.order_total_price AS total_price
    FROM raw_data r
    JOIN dim_customers c ON r.c_email = c.c_email
    JOIN dim_products p ON r.p_name = p.p_name AND r.p_brand = p.p_brand
    JOIN dim_stores s ON r.shop_name = s.shop_name
    JOIN dim_suppliers sup ON r.vendor_name = sup.vendor_name
""")
fact_sales.write.jdbc(url=pg_url, table="star_fact_sales", mode="overwrite", properties=pg_props)

print("Успех! Схема Звезда полностью создана и загружена в PostgreSQL.")

Собираем Таблицу Фактов (star_fact_sales)...
Успех! Схема Звезда полностью создана и загружена в PostgreSQL.


In [9]:
spark.stop()